### Cell 1 — Load the cleaned Day 1 output

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load the cleaned Day 1 output
df = pd.read_csv("../data/loans_clean.csv", low_memory=False)
print(f"Loaded: {df.shape}")
print(f"Class balance:\n{df['default_flag'].value_counts(normalize=True).round(3)}")

### Cell 2 — split into train / validation / test (60/20/20, stratified so class balance is preserved)

In [ ]:
# Separate target from features
y = df['default_flag']
X = df.drop(columns=['default_flag'])

# First split: 80% train+val vs 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Second split: from the 80%, take 75/25 → 60% train, 20% val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)

print(f"Train: {X_train.shape} — default rate {y_train.mean():.3f}")
print(f"Val:   {X_val.shape} — default rate {y_val.mean():.3f}")
print(f"Test:  {X_test.shape} — default rate {y_test.mean():.3f}")

### Cell 3 — look at what we're working with

In [ ]:
# Column dtype breakdown
print("Data types:")
print(X_train.dtypes.value_counts())
print()

# Cardinality of string columns (unique values per column)
str_cols = X_train.select_dtypes(include='object').columns
cardinality = X_train[str_cols].nunique().sort_values(ascending=False)
print("String columns by cardinality (top 20):")
print(cardinality.head(20))

### Cell 4 — drop the junk columns


In [ ]:
#Columns we know are IDs, free text, dates, or high-cardinality strings — drop them
drop_cols = [
    'id', 'member_id',
    'url', 'desc', 'title', 'emp_title',
    'zip_code',
    'earliest_cr_line', 'issue_d',
    'sec_app_earliest_cr_line', 'payment_plan_start_date',   # extra dates
    'addr_state',
]

X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val   = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])
X_test  = X_test.drop(columns=[c for c in drop_cols if c in X_test.columns])

# Backstop for anything still >50 unique values
high_card = [c for c in X_train.select_dtypes(exclude=np.number).columns
             if X_train[c].nunique() > 50]
X_train = X_train.drop(columns=high_card)
X_val   = X_val.drop(columns=high_card)
X_test  = X_test.drop(columns=high_card)

print(f"After pruning: {X_train.shape}")
print(f"Numeric features:     {X_train.select_dtypes(include=np.number).shape[1]}")
print(f"Categorical features: {X_train.select_dtypes(exclude=np.number).shape[1]}")

### Cell 6 — the WoE/IV function.

In [ ]:
def calculate_woe_iv(df, feature, target='default_flag', n_bins=5):
    """
    Compute Weight of Evidence per bin and total Information Value for a feature.
    Returns (iv_total, table_of_bin_stats). Returns (None, None) if computation fails.
    """
    data = df[[feature, target]].copy()

    # Bin the feature: quantile bins for numeric, raw values for categorical
    if pd.api.types.is_numeric_dtype(data[feature]):
        try:
            data['bin'] = pd.qcut(data[feature], q=n_bins, duplicates='drop').astype(str)
            data.loc[data[feature].isna(), 'bin'] = '__MISSING__'
        except (ValueError, TypeError):
            return None, None
    else:
        data['bin'] = data[feature].fillna('__MISSING__').astype(str)

    # Need at least 2 bins for WoE to make sense
    if data['bin'].nunique() < 2:
        return None, None

    # Count goods and bads per bin
    grouped = data.groupby('bin', observed=True)[target].agg(['sum', 'count'])
    grouped.columns = ['bad', 'total']
    grouped['good'] = grouped['total'] - grouped['bad']

    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()

    # Laplace smoothing — prevents log(0) if a bin has zero bads or zero goods
    eps = 0.5
    grouped['bad_dist']  = (grouped['bad']  + eps) / (total_bad  + eps * len(grouped))
    grouped['good_dist'] = (grouped['good'] + eps) / (total_good + eps * len(grouped))

    grouped['woe'] = np.log(grouped['good_dist'] / grouped['bad_dist'])
    grouped['iv']  = (grouped['good_dist'] - grouped['bad_dist']) * grouped['woe']

    return grouped['iv'].sum(), grouped

### Cell 7 — run it across all features and rank by IV

In [ ]:
# Combine training features with target (WoE is computed on the training set only)
train = X_train.copy()
train['default_flag'] = y_train.values

results = []
for col in X_train.columns:
    iv, _ = calculate_woe_iv(train, col)
    if iv is not None:
        results.append({'feature': col, 'iv': iv})

iv_df = pd.DataFrame(results).sort_values('iv', ascending=False).reset_index(drop=True)
print(f"Computed IV for {len(iv_df)} of {X_train.shape[1]} features\n")
print(iv_df.head(25).to_string())

### Cell 8 — Summarise how many features fall into each strength bracket

In [ ]:
strong     = iv_df[iv_df['iv'] > 0.3]
medium     = iv_df[(iv_df['iv'] > 0.1)  & (iv_df['iv'] <= 0.3)]
weak       = iv_df[(iv_df['iv'] > 0.02) & (iv_df['iv'] <= 0.1)]
useless    = iv_df[iv_df['iv'] <= 0.02]
suspicious = iv_df[iv_df['iv'] > 0.5]

print(f"Strong     (IV > 0.3):    {len(strong)} features")
print(f"Medium     (0.1 - 0.3):   {len(medium)} features")
print(f"Weak       (0.02 - 0.1):  {len(weak)} features")
print(f"Useless    (< 0.02):      {len(useless)} features — will drop")
print(f"Suspicious (> 0.5):       {len(suspicious)} features — check for leakage")

if len(suspicious) > 0:
    print("\nSuspicious features to inspect:")
    print(suspicious.to_string())

### Cell 9 — Apply the Pruning

In [ ]:
# Redundant / leakage-adjacent — Lending Club's own risk summary
drop_lc_grading = ['grade', 'sub_grade', 'int_rate']

# Drop one of two near-duplicate FICO columns
drop_redundant = ['fico_range_low']

# Weak features (IV < 0.02)
useless_features = iv_df[iv_df['iv'] < 0.02]['feature'].tolist()

# Combine everything to drop
final_drop = drop_lc_grading + drop_redundant + useless_features
final_drop = list(set([c for c in final_drop if c in X_train.columns]))

X_train = X_train.drop(columns=final_drop)
X_val   = X_val.drop(columns=final_drop)
X_test  = X_test.drop(columns=final_drop)

print(f"Dropped: {len(final_drop)} features")
print(f"  - {len(drop_lc_grading)} Lending Club grading columns (avoid trivial prediction)")
print(f"  - {len(drop_redundant)} redundant FICO column")
print(f"  - {len([c for c in useless_features if c in final_drop]) - len(drop_lc_grading) - len(drop_redundant) if False else len(useless_features)} low-IV features (IV < 0.02)")
print(f"\nKept: {X_train.shape[1]} features")

# Rebuild the IV ranking on the kept features only (for use in encoding step)
kept_iv = iv_df[iv_df['feature'].isin(X_train.columns)].reset_index(drop=True)
print(f"\nTop 15 kept features by IV:")
print(kept_iv.head(15).to_string())

### Cell 10 — Define the Encoder (defines two functions, no output)

In [ ]:
def fit_woe_encoder(train_df, target='default_flag', n_bins=5):
    """
    Learn bin edges + WoE values from the training set.
    Returns a dict of encoders — one per feature.
    """
    encoders = {}
    features = [c for c in train_df.columns if c != target]

    for col in features:
        data = train_df[[col, target]].copy()

        # Get bin labels
        if pd.api.types.is_numeric_dtype(data[col]):
            try:
                _, edges = pd.qcut(data[col].dropna(), q=n_bins,
                                   duplicates='drop', retbins=True)
            except (ValueError, TypeError):
                continue
            edges[0], edges[-1] = -np.inf, np.inf   # catch out-of-range val/test values
            data['bin'] = pd.cut(data[col], bins=edges, include_lowest=True).astype(str)
            data.loc[data[col].isna(), 'bin'] = '__MISSING__'
            col_type, saved_edges = 'numeric', edges
        else:
            data['bin'] = data[col].fillna('__MISSING__').astype(str)
            col_type, saved_edges = 'categorical', None

        # Compute WoE per bin (Laplace smoothing)
        g = data.groupby('bin', observed=True)[target].agg(['sum', 'count'])
        g.columns = ['bad', 'total']
        g['good'] = g['total'] - g['bad']
        eps = 0.5
        g['bad_dist']  = (g['bad']  + eps) / (g['bad'].sum()  + eps * len(g))
        g['good_dist'] = (g['good'] + eps) / (g['good'].sum() + eps * len(g))
        g['woe'] = np.log(g['good_dist'] / g['bad_dist'])

        encoders[col] = {'type': col_type, 'edges': saved_edges,
                         'woe_map': g['woe'].to_dict()}
    return encoders


def transform_woe(df, encoders):
    """Apply fitted WoE encoders to any dataframe (train/val/test)."""
    out = pd.DataFrame(index=df.index)
    for col, enc in encoders.items():
        if col not in df.columns:
            continue
        if enc['type'] == 'numeric':
            binned = pd.cut(df[col], bins=enc['edges'], include_lowest=True).astype(str)
            binned = binned.where(df[col].notna(), '__MISSING__')
        else:
            binned = df[col].fillna('__MISSING__').astype(str)
        out[col] = binned.map(enc['woe_map']).fillna(0.0)
    return out

### Cell 11 — fit on train, transform all three sets

In [ ]:
# Fit encoders on training data (with target)
train_combined = X_train.copy()
train_combined['default_flag'] = y_train.values
encoders = fit_woe_encoder(train_combined)
print(f"Fitted WoE encoders for {len(encoders)} features")

# Apply to train/val/test
X_train_woe = transform_woe(X_train, encoders)
X_val_woe   = transform_woe(X_val, encoders)
X_test_woe  = transform_woe(X_test, encoders)

print(f"\nTrain encoded: {X_train_woe.shape}")
print(f"Val encoded:   {X_val_woe.shape}")
print(f"Test encoded:  {X_test_woe.shape}")

print(f"\nAny NaN? train={X_train_woe.isna().any().any()}, "
      f"val={X_val_woe.isna().any().any()}, test={X_test_woe.isna().any().any()}")

print("\nSample of encoded features (first 5 rows, first 5 features):")
print(X_train_woe.iloc[:5, :5].round(3))

### Cell 12 — save encoded datasets so we can just load them:

In [ ]:
X_train_woe.assign(default_flag=y_train.values).to_csv("../data/train_woe.csv", index=False)
X_val_woe.assign(default_flag=y_val.values).to_csv("../data/val_woe.csv", index=False)
X_test_woe.assign(default_flag=y_test.values).to_csv("../data/test_woe.csv", index=False)
print("Saved encoded datasets to data/train_woe.csv, val_woe.csv, test_woe.csv")

### Cell 13 — save the encoders so we doesn't need to recompute them

In [ ]:
import pickle

with open("../data/woe_encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
print("Saved encoders to data/woe_encoders.pkl")

### Cell 14 — save a human-readable feature dictionary

In [ ]:
from pathlib import Path

kept_iv = iv_df[iv_df['feature'].isin(X_train.columns)].sort_values('iv', ascending=False)

lines = [
    "# Feature Dictionary\n",
    f"After WoE binning and pruning, **{len(kept_iv)} features** survived (IV >= 0.02, excluding Lending Club's own grading columns to force the model to learn from underlying data).\n",
    "## Information Value strength brackets\n",
    "| Bracket | IV range | Meaning |",
    "|---|---|---|",
    "| Strong | > 0.3 | Highly predictive |",
    "| Medium | 0.1 - 0.3 | Solid |",
    "| Weak | 0.02 - 0.1 | Marginal but usable |",
    "| Dropped | < 0.02 | Uninformative — removed |",
    "\n## Features ranked by IV\n",
    "| Rank | Feature | IV | Strength |",
    "|---|---|---|---|",
]
for i, row in kept_iv.reset_index(drop=True).iterrows():
    iv = row['iv']
    strength = ("Strong" if iv > 0.3 else "Medium" if iv > 0.1
                else "Weak"  if iv > 0.02 else "Dropped")
    lines.append(f"| {i+1} | `{row['feature']}` | {iv:.4f} | {strength} |")

Path("../docs").mkdir(exist_ok=True)
Path("../docs/feature_dictionary.md").write_text("\n".join(lines))
print("Saved to docs/feature_dictionary.md")